# TCC — Etapa 2: Ataque Adversarial Principal (CICIoT2023)

**Pergunta de pesquisa desta etapa:** como diferentes modelos clássicos de
Machine Learning (Decision Tree, Random Forest, LinearSVC) se comportam
diante de exemplos adversariais, quando os mesmos modelos já treinados e
avaliados na Etapa 1 são "congelados" e submetidos a um único método de
ataque, aplicado de forma consistente aos três?

```
CICIoT2023
     ↓
Modelos congelados          (Etapa 1: DecisionTree, RandomForest, LinearSVC)
     ↓
Amostra do conjunto de teste (só ataques, corretamente detectados)
     ↓
Método adversarial          (HopSkipJump — ver seção 2 para a justificativa)
     ↓
X_adv
     ↓
DT / RF / SVM                (avaliação direta + transferibilidade cruzada)
     ↓
Métricas de robustez
```

Esta etapa **não** re-treina nada. Os três modelos vêm exatamente como
foram salvos ao final da Etapa 1 (`artifacts_baseline/modelo_*.joblib`) —
"congelados" no sentido de que seus parâmetros internos permanecem fixos;
a única coisa que muda é a entrada que recebem.

## 0. Configuração e parâmetros

In [ ]:
!pip install -q adversarial-robustness-toolbox

In [ ]:
# Em ambiente Kaggle/Colab, instale as dependências antes de executar:
#   pip install -q adversarial-robustness-toolbox joblib scikit-learn pandas numpy matplotlib seaborn

import gc
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 80)

# =====================================================================
# PARÂMETROS DO EXPERIMENTO  (documentar estes valores na metodologia)
# =====================================================================
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Onde a Etapa 1 salvou os artefatos (mesmo ARTIFACTS_DIR daquele notebook).
ARTIFACTS_DIR_ETAPA1 = Path("artifacts_baseline")

# Onde esta etapa salva seus próprios resultados.
ARTIFACTS_DIR = Path("artifacts_adversarial")
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Quantas amostras de ataque (tráfego malicioso, corretamente detectado
# pelos 3 modelos) serão efetivamente atacadas. HopSkipJump é um ataque de
# caixa-preta baseado em consultas -- custoso por amostra -- então o valor
# é mantido pequeno para viabilidade no Colab. Aumentar apenas se o tempo
# de execução permitir.
N_SAMPLES_ATTACK = 60

# ---- Hiperparâmetros do HopSkipJump (art.attacks.evasion.HopSkipJump) ----
# max_iter: nº de iterações do algoritmo de busca binária + estimativa de
#   gradiente por amostra. Mais iterações = perturbação menor, porém mais
#   consultas ao modelo.
HSJ_MAX_ITER = 20
# max_eval / init_eval: nº de avaliações usadas para estimar a direção do
#   gradiente em cada iteração. Valores menores reduzem custo computacional
#   às custas de uma estimativa mais ruidosa.
HSJ_MAX_EVAL = 500
HSJ_INIT_EVAL = 100

# Se True, após o ataque as amostras adversariais são projetadas de volta
# para valores plausíveis de tráfego de rede (ver seção 6): recortadas aos
# limites observados no treino e arredondadas nas características que só
# assumem valores inteiros no dataset original.
ENFORCE_PLAUSIBILITY = True

print(f"N_SAMPLES_ATTACK = {N_SAMPLES_ATTACK}")
print(f"HopSkipJump: max_iter={HSJ_MAX_ITER}, max_eval={HSJ_MAX_EVAL}, init_eval={HSJ_INIT_EVAL}")


## 1. Carregar artefatos congelados da Etapa 1

In [ ]:
# ---------- Metadados e estatísticas de treino ----------
with open(ARTIFACTS_DIR_ETAPA1 / "metadados.json") as fh:
    META = json.load(fh)

FEATURE_NAMES = META["feature_names"]
POS_LABEL = META["pos_label"]  # 1 = Attack
print(f"Rótulo positivo (Attack): {POS_LABEL}")
print(f"Nº de características originais: {len(FEATURE_NAMES)}")

feature_stats = pd.read_csv(ARTIFACTS_DIR_ETAPA1 / "estatisticas_features_treino.csv", index_col=0)
feature_stats = feature_stats.loc[FEATURE_NAMES]  # garante a mesma ordem de X_test

# ---------- Conjunto de teste (mesmo split isolado da Etapa 1) ----------
npz = np.load(ARTIFACTS_DIR_ETAPA1 / "conjunto_teste.npz")
X_test = npz["X_test"].astype(np.float64)
y_test = npz["y_test"]
print(f"X_test: {X_test.shape}  |  y_test: {y_test.shape}  |  "
      f"proporção Attack no teste: {(y_test == POS_LABEL).mean():.4f}")

# ---------- Modelos congelados (pipelines completas: imputer→scaler→selector→clf) ----------
FROZEN_MODELS = {}
for path in sorted(ARTIFACTS_DIR_ETAPA1.glob("modelo_*.joblib")):
    name = path.stem.replace("modelo_", "")
    FROZEN_MODELS[name] = joblib.load(path)
    print(f"Carregado: {name}  <-  {path.name}")

assert len(FROZEN_MODELS) == 3, f"Esperava 3 modelos, encontrei {len(FROZEN_MODELS)}: {list(FROZEN_MODELS)}"

# ---------- Checagem de sanidade: a acurácia bate com a Etapa 1? ----------
print("\n=== Sanidade: desempenho dos modelos carregados no teste completo ===")
for name, pipe in FROZEN_MODELS.items():
    acc = pipe.score(X_test, y_test)
    print(f"{name:<14} accuracy(teste completo) = {acc:.4f}")


## 2. Metodologia do ataque adversarial

**Por que um único método precisa ser tecnicamente compatível com os três
modelos, e por que isso restringe a escolha.** Decision Tree e Random
Forest não são diferenciáveis — não existe gradiente da saída em relação à
entrada, então qualquer ataque *baseado em gradiente* (ex.: FGSM, PGD,
C&W) simplesmente não se aplica a eles sem recorrer a um modelo substituto
(*surrogate*), o que mudaria o experimento (estaríamos atacando um
substituto, não os modelos reais). O LinearSVC, por sua vez, tem gradiente
trivial (é linear), mas não expõe `predict_proba` — só `decision_function`.

**Método escolhido: HopSkipJump** (`art.attacks.evasion.HopSkipJump`,
Chen et al., 2020, Adversarial Robustness Toolbox). É um ataque de
**caixa-preta baseado em decisão**: não usa gradientes nem probabilidades,
apenas o rótulo de saída do modelo (`predict`) para uma sequência de
consultas, buscando iterativamente a menor perturbação que muda a
classificação. Isso o torna **igualmente aplicável a DecisionTree,
RandomForest e LinearSVC sem nenhuma adaptação por modelo** — a mesma
chamada de ataque, com os mesmos hiperparâmetros, funciona nos três,
o que valida a premissa de "avaliar o mesmo método nos três modelos"
proposta para este experimento.

**Como cada modelo é "consultado".** Cada `Pipeline` completa da Etapa 1
(imputer → scaler → selector → classificador) é envolvida por um
`SklearnClassifier` do ART, que trata a pipeline inteira como uma caixa
preta: o ataque manipula sempre as características **originais** (as
mesmas 39 do CICIoT2023), e é a própria pipeline congelada que aplica
imputação, padronização e seleção de características internamente a cada
consulta — exatamente como aconteceria com tráfego real chegando ao
sistema de detecção.

**Direção do ataque: evasão (não-alvo).** O objetivo do adversário é fazer
tráfego **malicioso** (`Attack`, rótulo 1) ser classificado como
**legítimo** (`Normal`, rótulo 0) — cenário de evasão de IDS, coerente com
a pergunta do TCC. Por isso a amostra de ataque (seção 3) é retirada
exclusivamente de tráfego de ataque real, corretamente detectado antes da
perturbação.

**O que será medido (seção 7):**
- **Taxa de sucesso direta**: para o ataque gerado contra o modelo M,
  qual fração das amostras deixa de ser detectada por M?
- **Transferibilidade**: um exemplo adversarial gerado especificamente
  contra o modelo M também engana os outros dois modelos? Isso testa se
  os três modelos compartilham vulnerabilidades semelhantes ou se cada
  um exige um ataque específico.
- **Magnitude da perturbação** (normas L2 e L∞): quão "grande" foi a
  mudança necessária no tráfego para conseguir a evasão — perturbações
  menores indicam vulnerabilidades mais graves (mais fáceis de explorar
  na prática).

## 3. Seleção da amostra de ataque

A amostra é retirada de `X_test` (nunca vista durante treino/seleção de
hiperparâmetros) e restrita a exemplos que sejam **simultaneamente**:

1. tráfego de **ataque real** (`y_test == POS_LABEL`);
2. **corretamente classificados como Attack pelos três modelos** antes de
   qualquer perturbação.

A condição (2) garante uma base de comparação justa: os três ataques
partem exatamente da mesma amostra-base, e qualquer evasão observada
depois é atribuível à perturbação — não a um exemplo que já era
mal-classificado de saída.

In [ ]:
is_attack = (y_test == POS_LABEL)

correct_all = np.ones(len(y_test), dtype=bool)
for name, pipe in FROZEN_MODELS.items():
    correct_all &= (pipe.predict(X_test) == y_test)

eligible_mask = is_attack & correct_all
eligible_idx = np.where(eligible_mask)[0]
print(f"Amostras de Attack no teste: {int(is_attack.sum()):,}")
print(f"Corretamente detectadas pelos 3 modelos: {len(eligible_idx):,} "
      f"({len(eligible_idx) / max(is_attack.sum(), 1):.1%} do total de ataques)")

rng = np.random.RandomState(RANDOM_STATE)
chosen_idx = rng.choice(eligible_idx, size=min(N_SAMPLES_ATTACK, len(eligible_idx)), replace=False)

X_sample = X_test[chosen_idx].copy()
y_sample = y_test[chosen_idx].copy()  # todos == POS_LABEL, por construção

print(f"\nAmostra de ataque selecionada: {X_sample.shape[0]} exemplos "
      f"({X_sample.shape[1]} características)")


## 4. Congelamento dos modelos como classificadores ART

Cada `Pipeline` é envolvida por `SklearnClassifier` (ART), com
`clip_values` definido **por característica**, a partir dos limites
mínimo/máximo observados no **treino** (`estatisticas_features_treino.csv`
da Etapa 1) — nunca do teste, para não introduzir informação do conjunto de
avaliação na própria definição do espaço de busca do ataque.

In [ ]:
from art.estimators.classification import SklearnClassifier

clip_min = feature_stats["min"].to_numpy()
clip_max = feature_stats["max"].to_numpy()

ART_CLASSIFIERS = {}
for name, pipe in FROZEN_MODELS.items():
    ART_CLASSIFIERS[name] = SklearnClassifier(model=pipe, clip_values=(clip_min, clip_max))
    print(f"{name:<14} -> {type(ART_CLASSIFIERS[name]).__name__}")


## 5. Geração dos exemplos adversariais (HopSkipJump)

O ataque é executado **uma vez por modelo**, sempre a partir da mesma
`X_sample` (seção 3) — ou seja, `X_adv_raw["DecisionTree"]` é a
perturbação otimizada especificamente contra a árvore, `X_adv_raw["RandomForest"]`
contra a floresta, e assim por diante. A avaliação cruzada
(transferibilidade) acontece depois, na seção 7.

In [ ]:
from art.attacks.evasion import HopSkipJump

X_adv_raw = {}
attack_time = {}

for name, art_clf in ART_CLASSIFIERS.items():
    print(f"\n{'=' * 60}\nHopSkipJump contra {name} ({len(X_sample)} amostras)...")
    attack = HopSkipJump(
        classifier=art_clf,
        targeted=False,          # evasão: só precisa deixar de ser "Attack"
        norm=2,
        max_iter=HSJ_MAX_ITER,
        max_eval=HSJ_MAX_EVAL,
        init_eval=HSJ_INIT_EVAL,
        verbose=False,
    )
    t0 = time.time()
    X_adv_raw[name] = attack.generate(x=X_sample)
    attack_time[name] = time.time() - t0
    print(f"  concluído em {attack_time[name]:.1f}s "
          f"({attack_time[name] / len(X_sample):.2f}s/amostra)")

    del attack
    gc.collect()


## 6. Restrição de plausibilidade (pós-processamento)

O HopSkipJump já respeita os limites mín/máx por característica
(`clip_values`, seção 4), mas não sabe que várias características do
CICIoT2023 só assumem **valores inteiros** no tráfego real (contagens de
flags, contagens de pacotes etc. — ver `is_integer_valued` nas estatísticas
da Etapa 1). Sem essa correção, o "ataque" poderia depender de, por
exemplo, `syn_count = 4.37`, que não corresponde a nenhum pacote de rede
real e tornaria o resultado inútil como estudo de robustez prático.

Esta etapa arredonda essas características e reaplica o recorte aos
limites observados, então **reavalia** as predições sobre a versão
projetada — as métricas da seção 7 usam essa versão plausível, não a saída
bruta do ataque.

In [ ]:
integer_cols = feature_stats.index[feature_stats["is_integer_valued"]].tolist()
integer_idx = [FEATURE_NAMES.index(c) for c in integer_cols]
print(f"Características tratadas como inteiras: {len(integer_idx)} de {len(FEATURE_NAMES)}")

X_adv = {}
for name, X_a in X_adv_raw.items():
    X_proj = X_a.copy()
    if ENFORCE_PLAUSIBILITY:
        X_proj[:, integer_idx] = np.round(X_proj[:, integer_idx])
        X_proj = np.clip(X_proj, clip_min, clip_max)
    X_adv[name] = X_proj

# Quanto a projeção de plausibilidade alterou o resultado do ataque?
for name in X_adv:
    pred_bruto = FROZEN_MODELS[name].predict(X_adv_raw[name])
    pred_proj = FROZEN_MODELS[name].predict(X_adv[name])
    mudou = (pred_bruto != pred_proj).sum()
    print(f"{name:<14} predições alteradas pela projeção de plausibilidade: "
          f"{mudou}/{len(pred_bruto)}")


## 7. Avaliação — robustez direta e transferibilidade

Para cada par (modelo-alvo do ataque, modelo avaliado), calculamos a
**taxa de sucesso de evasão (ASR)**: fração das amostras — originalmente
`Attack`, corretamente detectadas — que passam a ser classificadas como
`Normal` depois da perturbação.

- A **diagonal** da matriz é a robustez **direta**: o modelo M avaliado
  com o ataque desenhado especificamente contra M.
- As **células fora da diagonal** medem **transferibilidade**: o ataque
  foi otimizado contra um modelo, mas é aplicado a outro, sem nenhum
  ajuste — testa se os modelos compartilham a mesma superfície de
  vulnerabilidade.

In [ ]:
model_names = list(FROZEN_MODELS.keys())
asr_matrix = pd.DataFrame(index=model_names, columns=model_names, dtype=float)

for attack_source in model_names:
    X_a = X_adv[attack_source]
    for eval_model in model_names:
        pred = FROZEN_MODELS[eval_model].predict(X_a)
        evasion_rate = (pred != POS_LABEL).mean()  # deixou de ser detectado como Attack
        asr_matrix.loc[attack_source, eval_model] = evasion_rate

asr_matrix.index.name = "Ataque gerado contra ->"
asr_matrix.columns.name = "Avaliado em ->"
print("=== Taxa de sucesso de evasão (ASR) — linhas: origem do ataque | colunas: modelo avaliado ===")
display((asr_matrix * 100).round(1))

plt.figure(figsize=(5.5, 4.5))
sns.heatmap((asr_matrix * 100).astype(float), annot=True, fmt=".1f", cmap="Reds",
            vmin=0, vmax=100, cbar_kws={"label": "ASR (%)"})
plt.title("Transferibilidade dos ataques (HopSkipJump)\ndiagonal = robustez direta")
plt.xlabel("Modelo avaliado")
plt.ylabel("Ataque gerado contra")
plt.tight_layout()
plt.show()


## 8. Métricas de robustez — tabela consolidada

In [ ]:
def perturbation_norms(X_orig, X_pert):
    diff = (X_pert - X_orig).reshape(len(X_orig), -1)
    l2 = np.linalg.norm(diff, axis=1)
    linf = np.abs(diff).max(axis=1)
    return l2, linf

rows = []
for name in model_names:
    l2, linf = perturbation_norms(X_sample, X_adv[name])
    transfer_cols = [m for m in model_names if m != name]
    rows.append({
        "Modelo": name,
        "ASR direta (%)": 100 * asr_matrix.loc[name, name],
        "ASR média transferida (%)": 100 * asr_matrix.loc[name, transfer_cols].mean(),
        "Perturbação L2 (média)": l2.mean(),
        "Perturbação L2 (mediana)": np.median(l2),
        "Perturbação L_inf (média)": linf.mean(),
        "Tempo de ataque (s)": attack_time[name],
        "Tempo por amostra (s)": attack_time[name] / len(X_sample),
        "N amostras atacadas": len(X_sample),
    })

robustness_table = pd.DataFrame(rows).set_index("Modelo").round(4)
print("=== Métricas de robustez consolidadas ===")
display(robustness_table)

plt.figure(figsize=(6, 4))
robustness_table[["ASR direta (%)", "ASR média transferida (%)"]].plot(
    kind="bar", ax=plt.gca(), color=["#c0392b", "#e67e22"])
plt.ylabel("Taxa de sucesso de evasão (%)")
plt.title("Robustez direta vs. transferibilidade média por modelo")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 9. Persistência dos resultados

In [ ]:
robustness_table.to_csv(ARTIFACTS_DIR / "robustez_tabela_consolidada.csv")
asr_matrix.to_csv(ARTIFACTS_DIR / "asr_matriz_transferibilidade.csv")

np.savez_compressed(
    ARTIFACTS_DIR / "exemplos_adversariais.npz",
    X_sample=X_sample,
    **{f"X_adv_{name}": X_adv[name] for name in model_names},
    y_sample=y_sample,
    chosen_idx=chosen_idx,
)

ADV_META = {
    "n_samples_attack": int(len(X_sample)),
    "attack_method": "HopSkipJump (ART)",
    "attack_params": {
        "max_iter": HSJ_MAX_ITER, "max_eval": HSJ_MAX_EVAL, "init_eval": HSJ_INIT_EVAL,
        "norm": 2, "targeted": False,
    },
    "enforce_plausibility": ENFORCE_PLAUSIBILITY,
    "random_state": RANDOM_STATE,
    "attack_time_s": attack_time,
}
with open(ARTIFACTS_DIR / "metadados_adversarial.json", "w") as fh:
    json.dump(ADV_META, fh, indent=2)

print(f"Artefatos salvos em: {ARTIFACTS_DIR.resolve()}")
for p in sorted(ARTIFACTS_DIR.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size / 1024:.1f} KB)")


## 10. Discussão

*(preencher depois de rodar com os números efetivamente obtidos)*

- **Qual modelo foi mais robusto ao ataque direto?** Comparar a coluna
  "ASR direta (%)" da seção 8 — menor ASR = mais robusto ao HopSkipJump
  especificamente desenhado contra ele.
- **Há transferibilidade relevante entre os modelos?** Se a "ASR média
  transferida" for próxima da "ASR direta", os três modelos compartilham
  vulnerabilidades semelhantes apesar de arquiteturas muito diferentes
  (árvore, ensemble de árvores, linear) — um resultado interessante para
  discutir nas conclusões do TCC. Se for muito menor, cada modelo tem uma
  superfície de decisão mais idiossincrática.
- **A magnitude da perturbação (L2/L∞) é compatível com tráfego real?**
  Vale comparar esses valores com a escala típica de cada característica
  (`estatisticas_features_treino.csv`, Etapa 1) para julgar se a
  perturbação necessária seria realmente executável por um atacante, ou se
  é grande demais para ser prática — uma limitação a discutir.
- **Limitações desta etapa**: (i) `N_SAMPLES_ATTACK` é pequeno por
  restrição computacional — os resultados são indicativos, não uma
  estimativa populacional; (ii) o HopSkipJump ataca apenas o cenário de
  evasão não-direcionada (Attack → Normal); (iii) a restrição de
  plausibilidade (seção 6) é uma aproximação (limites observados + valores
  inteiros), não uma simulação completa de tráfego de rede real.